# HandTeX: Handwriting to LaTeX
Encoder-decoder models that convert handwritten math images to LaTeX strings.  
Dataset: [MathWriting-human](https://huggingface.co/datasets/deepcopy/MathWriting-human) (~230k samples).

In [ ]:
import torch
import numpy as np
import pickle
import sys
from pathlib import Path
from PIL import Image
from datasets import load_dataset

sys.path.insert(0, "src/models")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARTIFACTS = Path("src/artifacts")
print(f"Device: {DEVICE}")

## Load Dataset

In [ ]:
ds = load_dataset("deepcopy/MathWriting-human")
ds_test = ds["test"]
print(f"Test set: {len(ds_test)} samples")

# Show a few examples
for i in range(3):
    sample = ds_test[i]
    print(f"\nSample {i}: {sample['latex'][:80]}")
    display(sample["image"])

## Preprocessing & Tokenizer

In [ ]:
def resize_pad_grayscale(img_pil, target=256, pad_value=255):
    """Resize keeping aspect ratio, pad to square."""
    img = img_pil.convert("L")
    w, h = img.size
    scale = target / max(w, h)
    new_w, new_h = max(1, int(round(w * scale))), max(1, int(round(h * scale)))
    img_rs = img.resize((new_w, new_h), resample=Image.BICUBIC)
    canvas = Image.new("L", (target, target), color=pad_value)
    canvas.paste(img_rs, ((target - new_w) // 2, (target - new_h) // 2))
    return canvas

# Load tokenizer
with open(ARTIFACTS / "lstm_tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

VOCAB_SIZE = 66
START_TOKEN = VOCAB_SIZE - 2
END_TOKEN = VOCAB_SIZE - 1
inv_vocab = {v: k for k, v in tokenizer.word_index.items()}

def decode_tokens(seq):
    """Convert token ids back to a LaTeX string."""
    return "".join(inv_vocab.get(t, "") for t in seq if t not in (0, START_TOKEN, END_TOKEN))

print(f"Vocab size: {VOCAB_SIZE}")

In [ ]:
def normalized_edit_distance(s1, s2):
    """Levenshtein distance normalized by max length."""
    if len(s1) == 0 and len(s2) == 0: return 0.0
    if len(s1) == 0 or  len(s2) == 0: return 1.0
    d = [[0] * (len(s2) + 1) for _ in range(len(s1) + 1)]
    for i in range(len(s1) + 1): d[i][0] = i
    for j in range(len(s2) + 1): d[0][j] = j
    for i in range(1, len(s1) + 1):
        for j in range(1, len(s2) + 1):
            cost = 0 if s1[i-1] == s2[j-1] else 1
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+cost)
    return d[len(s1)][len(s2)] / max(len(s1), len(s2))

## Load Models

In [ ]:
# Model 1: DINOv2 encoder (LoRA) + LSTM decoder with Bahdanau attention
from vit_lora_lstm_attn import ViTLatexModelLoRA as LSTMModel

lstm_model = LSTMModel(vocab_size=VOCAB_SIZE, lora_r=16).to(DEVICE)
ckpt = torch.load(ARTIFACTS / "lstm.pt", map_location=DEVICE, weights_only=False)
lstm_model.load_state_dict(ckpt["model"])
lstm_model.eval()
print("LSTM model loaded.")

In [ ]:
# Model 2: DINOv2 encoder (LoRA) + Transformer decoder
from vit_transformer_v2 import ViTLatexModelLoRA as TransformerModel

transformer_model = TransformerModel(vocab_size=VOCAB_SIZE).to(DEVICE)
ckpt = torch.load(ARTIFACTS / "transformer.pt", map_location=DEVICE, weights_only=False)
transformer_model.load_state_dict(ckpt["model"])
transformer_model.eval()
print("Transformer model loaded.")

## Evaluation on Test Set

In [ ]:
# Tokenize ground truth (round-trip through our tokenizer for fair comparison)
from tensorflow.keras.preprocessing.sequence import pad_sequences

gt_raw = [s["latex"] for s in ds_test]
seqs = tokenizer.texts_to_sequences(gt_raw)
seqs = [[START_TOKEN] + s + [END_TOKEN] for s in seqs]
tokens_test = pad_sequences(seqs, padding="post")
gt_strings = [decode_tokens(t.tolist()) for t in tokens_test]

# Preprocess images
N = len(ds_test)
test_images = []
for i, s in enumerate(ds_test):
    img = resize_pad_grayscale(s["image"], target=256)
    img = np.array(img, dtype=np.float32) / 255.0
    test_images.append(img)
    if (i + 1) % 500 == 0 or i == N - 1:
        print(f"Preprocessed {i+1}/{N}", flush=True)

test_images = torch.tensor(np.array(test_images), dtype=torch.float32).unsqueeze(1)
print(f"Images shape: {test_images.shape}")

In [ ]:
models = {
    "LSTM (DINOv2 + LoRA)": lstm_model,
    "Transformer (DINOv2)": transformer_model,
}

results = {}

for model_name, model in models.items():
    use_beam = hasattr(model, "generate_beam")
    exact = 0
    total_ed = 0.0

    print(f"\n{model_name} ({N} samples)")
    print("-" * 50)

    for i in range(N):
        img = test_images[i:i+1].repeat(1, 3, 1, 1).to(DEVICE)
        gt = gt_strings[i]

        with torch.no_grad():
            if use_beam:
                pred_tokens = model.generate_beam(img, max_len=150, sos_idx=START_TOKEN, eos_idx=END_TOKEN, beam_size=5)
            else:
                pred_tokens = model.generate(img, max_len=150, sos_idx=START_TOKEN, eos_idx=END_TOKEN)

        pred = decode_tokens(pred_tokens)
        ed = normalized_edit_distance(pred, gt)
        if pred == gt:
            exact += 1
        total_ed += ed

        if (i + 1) % 100 == 0 or i == N - 1:
            print(f"  [{i+1:>5}/{N}] exact={exact/(i+1):.2%}, avg ED={total_ed/(i+1):.4f}", flush=True)

    results[model_name] = {
        "exact_match": exact / N,
        "avg_edit_distance": total_ed / N,
    }

## Results

In [ ]:
print(f"{'Model':<30} {'Exact Match':>12} {'Avg Edit Dist':>14}")
print("-" * 58)
for model_name, metrics in results.items():
    print(f"{model_name:<30} {metrics['exact_match']:>11.2%} {metrics['avg_edit_distance']:>14.4f}")